# Lecture 1: The brachistochrone, and what an optimal control problem is

**The challenge.** A bead slides without friction down a wire from a point
$A$ to a lower point $B$ that is not directly below it. What shape should the
wire have so that the bead arrives as soon as possible? Calculus teaches us to
minimize a *function*: pick the number $x$ that makes $f(x)$ smallest. Here
the unknown is a whole *curve*, infinitely many numbers at once, and the
quantity to minimize, the travel time, depends on all of them together. No
amount of setting a derivative to zero in one variable will find it.

This course is about problems of that kind, where the unknown is a function
of time: a steering history, a thrust program, a torque profile. They are
called **optimal control problems**. This first lecture poses one precisely,
in the form the rest of the course (and the package) uses, and solves it
without yet explaining how. By Lecture 15 every step of that solve will have
been taken apart.

**After this lecture you can**

- state an optimal control problem in Bolza form: states, controls, dynamics,
  costs, boundary conditions and control constraints;
- write one down with `OptimalControlProblem`, with runtime parameters,
  control sets and free or fixed final time;
- recognize the Lagrange, Mayer and Bolza forms and convert between them.

In [ ]:
# The challenge, drawn: three wires from A to B and where each bead is every 0.1 s.
import matplotlib.pyplot as plt
import numpy as np
from scipy.optimize import brentq

BLUE, ORANGE, GRAY = "#2a78d6", "#eb6834", "#8a8a85"
G, XF, YF = 9.81, 2.0, 1.0


def descent(xs, ys):
    """Time at each vertex of a polyline (y down), sliding from rest: exact for the polyline."""
    t, v = [0.0], 0.0
    for dx, dy in zip(np.diff(xs), np.diff(ys)):
        ds = np.hypot(dx, dy)
        v1 = np.sqrt(max(v * v + 2 * G * dy, 0.0))
        t.append(t[-1] + 2 * ds / (v + v1))
        v = v1
    return np.array(t)


s = np.linspace(0, 1, 4001)
phi_f = brentq(lambda f: (f - np.sin(f)) / (1 - np.cos(f)) - XF / YF, 1e-6, 2 * np.pi - 1e-6)
R = YF / (1 - np.cos(phi_f))
r = (XF**2 + YF**2) / (2 * XF)                     # circle tangent to the vertical at A
a_B = np.arctan2(YF, r - XF)                        # angle of B seen from the centre
wires = [
    ("straight line", XF * s, YF * s, ORANGE),
    ("arc of a circle (Galileo)", r - r * np.cos(a_B * s), r * np.sin(a_B * s), GRAY),
    ("cycloid (Bernoulli)", R * (phi_f * s - np.sin(phi_f * s)), R * (1 - np.cos(phi_f * s)), BLUE),
]

fig, ax = plt.subplots(figsize=(9, 4.6))
for name, xs, ys, color in wires:
    tv = descent(xs, ys)
    ax.plot(xs, ys, color=color, lw=2, label=f"{name}: {tv[-1]:.3f} s")
    ticks = np.arange(0.1, tv[-1], 0.1)
    ax.plot(np.interp(ticks, tv, xs), np.interp(ticks, tv, ys), "o", color=color, ms=7,
            mec="white", mew=1.5)
for (px, py), name in [((0, 0), "A"), ((XF, YF), "B")]:
    ax.plot(px, py, "o", color="k", ms=8)
    ax.annotate(name, (px, py), xytext=(-16, 4) if name == "A" else (9, -4),
                textcoords="offset points", fontsize=14, weight="bold")
ax.annotate("", xy=(2.25, 0.55), xytext=(2.25, 0.15),
            arrowprops={"arrowstyle": "-|>", "color": "k", "lw": 1.5})
ax.text(2.29, 0.35, "g", fontsize=13, style="italic", va="center")
ax.set(xlim=(-0.15, 2.45), ylim=(1.38, -0.12), xlabel="x (m)", ylabel="depth y (m)")
ax.set_aspect("equal")
ax.set_title("A bead slides from A to B without friction: each dot is 0.1 s",
             fontsize=11, loc="left")
ax.legend(frameon=False, fontsize=9, loc="upper right", bbox_to_anchor=(1.0, 0.98),
          title="wire: time from A to B", title_fontsize=9)
ax.grid(alpha=0.25)
fig.tight_layout()

*Three wires from $A$ to $B$, with each bead drawn every tenth of a second. The straight line is the shortest path and the slowest. A steeper start buys speed early, and the best wire balances that against the extra distance. Section 4 computes these times.*

## 1. A challenge to the sharpest minds of the world

In June 1696 Johann Bernoulli, professor at Groningen, printed a problem in
the Leipzig journal *Acta Eruditorum* and addressed it "to the most acute
mathematicians of the whole world". Find the curve of quickest descent, the
*brachistochrone* (Greek *brachistos*, shortest, and *chronos*, time).
Galileo had considered it in 1638 and guessed, wrongly, that the answer was an
arc of a circle. Bernoulli knew the true answer and was fishing for rivals,
above all his older brother Jakob, with whom he had a long and bitter
quarrel, and Isaac Newton, whom the continental mathematicians suspected of
claiming more than he could do.

Leibniz persuaded Bernoulli to extend the deadline to Easter 1697. The
problem reached Newton, by then Warden of the Royal Mint in London, on 29
January 1697. According to his niece Catherine Conduitt, who kept house for
him, he came home from the Mint at four in the afternoon tired, found the
letter, and did not go to bed until he had solved it, around four in the
morning. He sent the answer to the Royal Society, which printed it
anonymously. Bernoulli was not fooled; he is supposed to have said that he
recognized the author *tanquam ex ungue leonem*, "as the lion by its claw".

Five solutions arrived: from Newton, Leibniz, Jakob Bernoulli, l'Hôpital and
Tschirnhaus, besides Johann's own. Johann's was the most beautiful. He treated
the falling bead as a ray of light passing through layers of glass whose
speed of light increases with depth. Light takes the quickest path (Fermat's
principle), and it obeys Snell's law at each layer:

$$
\frac{\sin\theta}{v} = \text{const}, \qquad v = \sqrt{2 g y},
$$

where $\theta$ is the angle of the path from the vertical and $y$ is the
depth fallen. The curve with that property is a **cycloid**, the curve traced
by a point on the rim of a rolling wheel. Jakob's solution was clumsier but
more general: he varied a small piece of the curve and asked how the time
changed. That idea, varying the unknown function and requiring the time not
to change to first order, became the **calculus of variations**, the subject
of Lecture 2. Its descendant, Pontryagin's minimum principle, runs underneath
everything in this package.

The brothers never reconciled. Their quarrel over who had solved which
problem first went on until Jakob's death in 1705.

In [ ]:
# Bernoulli's two pictures: light bending through faster and faster layers, and the
# wheel whose rim traces the answer.
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.patches import Circle
import itertools

BLUE, ORANGE, GRAY = "#2a78d6", "#eb6834", "#8a8a85"
fig, (left, right) = plt.subplots(1, 2, figsize=(11, 4.2))

# Left: layers of glass, speed growing with depth like sqrt(depth); Snell: sin(theta)/v const.
n_layers, depth = 8, 1.0
edges = np.linspace(0, depth, n_layers + 1)
mid = (edges[:-1] + edges[1:]) / 2
speed = np.sqrt(mid)
k = np.sin(np.radians(12)) / speed[0]                 # sin(theta)/v on the first layer
x, y = [0.0], [0.0]
for (top, bottom), v in zip(itertools.pairwise(edges), speed):
    left.axhspan(top, bottom, color=BLUE, alpha=0.06 + 0.5 * v / speed[-1] * 0.5, lw=0)
    th = np.arcsin(min(k * v, 0.999))
    x.append(x[-1] + (bottom - top) * np.tan(th))
    y.append(bottom)
left.plot(x, y, color=ORANGE, lw=2.5, marker="o", ms=4)
for i in (0, 4, 7):
    x0, y0 = x[i], y[i]
    left.plot([x0, x0], [y0, y0 + 0.11], color="k", lw=0.8, ls=":")
    left.text(x0 + 0.025, y0 + 0.075, rf"$\theta_{i + 1}$", fontsize=10)
left.text(1.35, 0.06, "slow", fontsize=10, color="k", ha="right")
left.text(1.35, 0.97, "fast", fontsize=10, color="k", ha="right")
left.set(xlim=(-0.05, 1.4), ylim=(1.0, 0.0), xlabel="x", ylabel="depth")
left.set_title(r"light through layers: $\sin\theta / v$ is the same in every layer",
               fontsize=10, loc="left")
left.set_aspect("equal")

# Right: a wheel rolling along the underside of the line y = 0 traces the cycloid.
Rw = 0.5
phi = np.linspace(0, 2 * np.pi, 400)
right.plot(Rw * (phi - np.sin(phi)), Rw * (1 - np.cos(phi)), color=BLUE, lw=2.5)
right.axhline(0, color="k", lw=1.2)
for p_, alpha in [(0.9, 0.35), (2.0, 0.6), (3.4, 1.0)]:
    cx = Rw * p_
    right.add_patch(Circle((cx, Rw), Rw, fill=False, color=GRAY, lw=1.2, alpha=alpha))
    px, py = Rw * (p_ - np.sin(p_)), Rw * (1 - np.cos(p_))
    right.plot([cx, px], [Rw, py], color=GRAY, lw=1, alpha=alpha)
    right.plot(px, py, "o", color=ORANGE, ms=7, alpha=alpha)
right.annotate("", xy=(2.3, 0.5), xytext=(1.9, 0.5),
               arrowprops={"arrowstyle": "-|>", "color": "k", "lw": 1})
right.text(2.1, 0.43, "rolls", fontsize=9, ha="center")
right.set(xlim=(-0.1, 2 * np.pi * Rw + 0.1), ylim=(1.15, -0.1), xlabel="x", ylabel="depth")
right.set_title("a point on a rolling wheel traces a cycloid", fontsize=10, loc="left")
right.set_aspect("equal")
for a in (left, right):
    a.set_xticks([])
    a.set_yticks([])
fig.tight_layout()

*Bernoulli's two pictures. Light crossing layers in which it travels faster and faster bends towards the horizontal, keeping $\sin\theta / v$ fixed. A falling bead's speed grows as $\sqrt{2 g y}$, like the light's, and the curve with that property is the cycloid, traced by a point on a rolling wheel.*

## 2. The problem as a control problem

Bernoulli's question is about a curve. We restate it as a question about a
**control**: at each instant the bead is steered, by the wire, in some
direction. Let $(x, y)$ be its position, with $y$ pointing *down*, and $v$ its
speed. If the wire points along the unit vector $a = (a_x, a_y)$, the bead
moves along it, and gravity speeds it up by the component of $g$ along it:

$$
\dot x = v\,a_x, \qquad \dot y = v\,a_y, \qquad \dot v = g\,a_y,
\qquad a_x^2 + a_y^2 = 1 .
$$

It starts at rest at the origin and must reach $(x_f, y_f)$; when it gets
there is up to us, and we want it as soon as possible:

$$
\min\; t_f = \int_0^{t_f} 1 \, dt .
$$

That is the shape of every problem in this course:

- **states** $x(t) \in \mathbb{R}^n$ (here $x, y, v$), whose evolution is
  fixed once the controls are;
- **controls** $u(t)$ (here the direction $a$), which we choose, often
  within a set $U$ (here the unit circle);
- **dynamics** $\dot x = f(t, x, u)$;
- **boundary conditions**: initial states, and terminal constraints
  $\psi(t_f, x(t_f)) = 0$;
- a **cost** to minimize.

In [ ]:
# The state and the control on one bead: position (x, y), speed v along the wire,
# direction a, and the part of gravity that acts along it.
import matplotlib.pyplot as plt
import numpy as np

BLUE, ORANGE, GRAY = "#2a78d6", "#eb6834", "#8a8a85"
fig, ax = plt.subplots(figsize=(7, 4.4))
theta = np.radians(55)                               # direction of the wire from straight down
a = np.array([np.sin(theta), np.cos(theta)])         # (a_x, a_y), y pointing down
P = np.array([1.0, 0.8])
s = np.linspace(-1.6, 1.6, 200)
curv = -0.18                                         # a gently curving wire through P
wire = P[:, None] + a[:, None] * s + np.array([-a[1], a[0]])[:, None] * curv * s**2
ax.plot(*wire, color=GRAY, lw=4, solid_capstyle="round", zorder=1)
arrow = {"arrowstyle": "-|>", "lw": 2.2, "mutation_scale": 16}
ax.annotate("", xy=P + 0.95 * a, xytext=P, arrowprops=dict(color=BLUE, **arrow), zorder=4)
ax.text(*(P + 0.95 * a + [-0.1, 0.12]), r"$a = (a_x, a_y)$: the control," "\n" "the direction of the wire",
        fontsize=10, color=BLUE, va="top")
ax.annotate("", xy=P + [0, 0.9], xytext=P, arrowprops=dict(color="k", **arrow), zorder=4)
ax.text(P[0] - 0.06, P[1] + 0.85, "g", fontsize=13, style="italic", ha="right")
off = 0.13 * np.array([a[1], -a[0]])                 # draw the component beside the wire
comp = 0.9 * a[1] * a
ax.annotate("", xy=P + off + comp, xytext=P + off, arrowprops=dict(color=ORANGE, **arrow))
ax.text(*(P + off + comp + [0.04, -0.05]), r"$g\,a_y = g\cos\theta$:" "\n" "gravity along the wire",
        fontsize=10, color=ORANGE, va="bottom")
arc = np.linspace(0, theta, 40)
ax.plot(P[0] + 0.45 * np.sin(arc), P[1] + 0.45 * np.cos(arc), color="k", lw=0.9)
ax.text(P[0] + 0.14, P[1] + 0.58, r"$\theta$", fontsize=12)
ax.plot(*P, "o", color=ORANGE, ms=18, mec="white", mew=2, zorder=5)
ax.text(P[0] - 0.3, P[1] + 0.05, "bead at $(x, y)$\nmoving at speed $v$", fontsize=10, ha="right")
ax.set(xlim=(-0.9, 3.0), ylim=(2.0, -0.2))
ax.set_aspect("equal")
ax.axis("off")
ax.set_title(r"$\dot x = v\,a_x, \quad \dot y = v\,a_y, \quad \dot v = g\,a_y, \quad a_x^2 + a_y^2 = 1$",
             fontsize=11)
fig.tight_layout()

*One bead, one instant. The state is where it is and how fast it moves; the control is the direction the wire points. Only the part of gravity along the wire changes the speed.*

### Bolza, Lagrange and Mayer

The most general cost we will use is the **Bolza** form,

$$
J = \phi\big(t_f, x(t_f)\big) + \int_{t_0}^{t_f} L(t, x, u)\,dt,
$$

with a *terminal cost* $\phi$ and a *running cost* $L$. Two special cases have
names: **Lagrange** ($\phi = 0$, as above, with $L = 1$) and **Mayer**
($L = 0$). They are equivalent. A running cost becomes a terminal one by
adding a state $\dot x_{n+1} = L$, $x_{n+1}(t_0) = 0$, and minimizing
$x_{n+1}(t_f)$. A terminal cost becomes a running one by writing
$\phi(t_f, x(t_f)) = \phi(t_0, x(t_0)) + \int \dot\phi\, dt$. So the choice is
a matter of convenience. The minimum-time brachistochrone is a Lagrange
problem with $L = 1$, or a Mayer problem with $\phi = t_f$.

## 3. Posing it for the package

`OptimalControlProblem` takes exactly these ingredients as SymPy
expressions. The constants $g$, $x_f$ and $y_f$ are declared as **parameters**:
they stay symbolic, so the necessary conditions are derived once and then
evaluated for any values. The final time is free because we do not pass
`final_time`. The control set `UnitVector([a_x, a_y])` says the controls form a
direction.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import sympy as sp

from auto_optimal_control.problem import Box, OptimalControlProblem, UnitVector

BLUE, ORANGE, GRAY = "#2a78d6", "#eb6834", "#8a8a85"

t = sp.Symbol("t", real=True)
x, y, v, ax, ay = sp.symbols("x y v a_x a_y", real=True)
g, xf, yf = sp.symbols("g x_f y_f", positive=True)

brachistochrone = OptimalControlProblem(
    name="brachistochrone",
    time=t,
    states=[x, y, v],
    controls=[ax, ay],
    params=[g, xf, yf],
    dynamics=[v * ax, v * ay, g * ay],
    running_cost=1,
    initial_state=[0, 0, 0],
    final_state={x: xf, y: yf},
    control_sets=[UnitVector([ax, ay])],
)
brachistochrone

The object checks what it is given. Every expression may use only the
symbols it declares, each state needs exactly one equation, and there cannot
be more terminal conditions than the problem has freedom for. A typo
surfaces immediately rather than as a wrong answer later:

In [ ]:
try:
    OptimalControlProblem(
        time=t, states=[x, y, v], controls=[ax, ay],
        dynamics=[v * ax, v * ay, sp.Symbol("G") * ay],   # G is not declared
        running_cost=1, initial_state=[0, 0, 0],
    )
except ValueError as err:
    print("ValueError:", err)

The pieces are all there as attributes. Note how `final_state` became
terminal constraints $\psi = x(t_f) - x_f = 0$, $y(t_f) - y_f = 0$, and that
`v` has no terminal condition: its final value is free.

In [ ]:
print("terminal constraints:", brachistochrone.terminal_constraints)
print("initial state:       ", brachistochrone.initial_state)
print("free final time:     ", brachistochrone.free_final_time)
print("parameters:          ", brachistochrone.params)

### Control sets

Controls are unconstrained unless they are placed in a set. There are two
kinds:

- `Box(u, lower, upper)`: bounds on one control. Bounds may be numbers,
  parameters or expressions. A force limit $|u| \le u_{\max}$ is
  `Box(u, -u_max, u_max)`.
- `UnitVector([u1, u2, ...])`: the controls form a direction. That fits a
  heading, a steering direction, or the pointing of a rocket engine.

What a control set does to the solution (clipping, bang-bang switching,
pointing along a "primer vector") is the subject of Lecture 4. One more
convention matters already: declare quantities that are positive (masses,
gravity, speeds of light) with `positive=True`. The derivation uses it to
simplify square roots and norms, and to decide which way inequalities go.

## 4. A first solve, as a black box

The rest of the course explains how to solve such a problem. Here is the
whole machinery used as a black box, so we can look at the answer first.
`Homotopy` derives the necessary conditions, compiles them to C and then
searches for an extremal. All it needs, beyond the problem and its parameter
values, is a range to draw guesses of the initial **costates** from (the
multipliers of Lecture 2; for now, three unknown numbers) and of $t_f$.

In [ ]:
from auto_optimal_control.homotopy import Homotopy

params = [9.81, 2.0, 1.0]          # g, x_f, y_f: 2 m across and 1 m down
solver = Homotopy(brachistochrone)
result = solver.solve(params, costates=([-1, -1, -1], [1, 1, 1]), final_time=(0.2, 3.0))
print(result.message)
print(f"t_f = {result.solution.tf:.12f} s after {result.tries} random draw(s)")

Now compare with Bernoulli. The cycloid through the origin and $(x_f, y_f)$ is
$x = R(\varphi - \sin\varphi)$, $y = R(1 - \cos\varphi)$, where the final
angle $\varphi_f$ solves
$(\varphi_f - \sin\varphi_f)/(1 - \cos\varphi_f) = x_f / y_f$. The descent
takes $t_f = \varphi_f\sqrt{R/g}$. For contrast, the straight line takes
$L\sqrt{2/(g\,y_f)}$, with $L$ its length.

In [ ]:
from scipy.optimize import brentq

G, XF, YF = params
phi_f = brentq(lambda f: (f - np.sin(f)) / (1 - np.cos(f)) - XF / YF, 1e-6, 2 * np.pi - 1e-6)
R = YF / (1 - np.cos(phi_f))
t_cycloid = phi_f * np.sqrt(R / G)
length = np.hypot(XF, YF)
t_line = length * np.sqrt(2 / (G * YF))
print(f"cycloid       t_f = {t_cycloid:.12f} s")
print(f"package       t_f = {result.solution.tf:.12f} s   "
      f"(difference {abs(result.solution.tf - t_cycloid):.1e})")
print(f"straight line t_f = {t_line:.6f} s, {100 * (t_line / t_cycloid - 1):.1f}% slower")

In [ ]:
tr = result.shooting.trajectory(result.solution.v, n_points=300)
phi = np.linspace(0, phi_f, 300)

fig, (left, right) = plt.subplots(1, 2, figsize=(11, 3.6))
left.plot(R * (phi - np.sin(phi)), R * (1 - np.cos(phi)), color=GRAY, lw=6, alpha=0.5,
          label="cycloid (Bernoulli, 1697)")
left.plot(tr["z"][:, 0], tr["z"][:, 1], color=BLUE, lw=2, label="extremal (package)")
left.plot([0, XF], [0, YF], color=ORANGE, lw=1.5, ls="--", label="straight line")
left.invert_yaxis()
left.set_aspect("equal")
left.set(xlabel="x (m)", ylabel="depth y (m)")
left.legend(frameon=False, fontsize=9, loc="upper right")
# The direction of motion, (x', y') = v a, measured from straight down.
speed = tr["z"][:, 2]
print(f"v at the end: {speed[-1]:.4f} m/s; a at the end: {np.round(tr['u'][-1], 4)}")
heading = np.degrees(np.arctan2(speed * tr["u"][:, 0], speed * tr["u"][:, 1]))
right.plot(tr["t"][1:], heading[1:], color=BLUE, lw=2)
right.set(xlabel="t (s)", ylabel="direction of motion from vertical (deg)")
right.set_title("the control: where the wire points", fontsize=10, loc="left")
for a in (left, right):
    a.grid(alpha=0.3)
fig.tight_layout()

The extremal *is* the cycloid, to about twelve digits. It starts straight
down (direction $0°$, to gain speed as fast as possible), sweeps through the
horizontal, and here even climbs at the end (past $90°$): the bead dips below
$B$ and comes back up, because speed gained early pays for the extra
distance. The direction turns at a constant rate, a fact Lecture 3 will
explain: along the extremal, $\sin\theta / v$ is constant, which is Snell's
law in another guise.

One detail of this particular extremal: it runs with $v < 0$ and the unit
vector $a$ pointing *backwards*. The dynamics cannot tell $(v, a)$ from
$(-v, -a)$, since both give the same motion $\dot x = v a$, so the necessary
conditions have both solutions and the random search found the mirrored one.
That is why we plot the direction of motion, $v\,a$, rather than $a$. Such
symmetries are common, and part of reading a result is recognizing them.

### The same problem in Mayer form

As a check on the equivalence of the forms, pose the problem with a terminal
cost $\phi = t_f$ and no running cost. It is a different problem object and a
different derivation, but it has the same solution.

In [ ]:
mayer = OptimalControlProblem(
    name="brachistochrone_mayer", time=t, states=[x, y, v], controls=[ax, ay],
    params=[g, xf, yf], dynamics=[v * ax, v * ay, g * ay],
    terminal_cost=t, initial_state=[0, 0, 0], final_state={x: xf, y: yf},
    control_sets=[UnitVector([ax, ay])],
)
r_mayer = Homotopy(mayer).solve(params, ([-1, -1, -1], [1, 1, 1]), (0.2, 3.0))
print(f"Mayer form t_f = {r_mayer.solution.tf:.12f} s")

## 5. What else a problem can say

The brachistochrone uses most of `OptimalControlProblem`. The remaining
options appear in later lectures; for reference:

| argument | meaning | first used |
|---|---|---|
| `terminal_cost` | $\phi(t_f, x(t_f))$, in `time` and the states | above; Lecture 16 (maximize a final radius) |
| `terminal_constraints` | extra $\psi(t_f, x(t_f)) = 0$, beyond `final_state` | Lecture 10 (reach a circular orbit) |
| `initial_state` | a list (`None` = free) or a dict; omitted states are free | Lecture 2 |
| `final_time` | fix $t_f$ (a number or a parameter expression) | Lecture 3 |
| `initial_time` | $t_0$, default 0 | |
| `control_sets` | `Box` and `UnitVector` | Lecture 4 |
| `path_constraints` | state limits $g(t, x) \le 0$ along the way | Lecture 13 |

The package also ships a set of textbook problems with known solutions in
`auto_optimal_control.examples`. We will meet most of them.

In [ ]:
from auto_optimal_control import examples

for name, problem in examples.all_examples().items():
    print(f"{name:34s} {problem!r}")

## Summary

- An optimal control problem chooses a function of time, the control, to
  minimize a cost, subject to dynamics, boundary conditions and constraints
  on the control.
- Bolza, Lagrange and Mayer forms are equivalent; the package takes the Bolza
  form, $\phi + \int L$.
- `OptimalControlProblem` holds the problem symbolically. Constants are
  runtime parameters; control sets are `Box` and `UnitVector`; the final time
  is free unless fixed.
- The brachistochrone's extremal is Bernoulli's cycloid, and the package
  finds it from nothing but a range of costate guesses. How it does that is
  the subject of the next fourteen lectures.

## Exercises

1. Solve for $(x_f, y_f) = (1, 1)$ and $(\pi, 2)$. For which endpoints does
   the bead dip below $B$ and come back up? (Hint: compare $\varphi_f$ with
   $\pi$.)
2. Galileo's guess was an arc of a circle. Compute the descent time along the
   quarter circle from $(0, 0)$ to $(1, 1)$ (tangent to the vertical at the
   start) by quadrature, $T = \int ds / \sqrt{2 g y}$, and compare it with the
   cycloid's and the straight line's.
3. Convert the brachistochrone to Lagrange form with an extra state: add
   $\dot c = 1$, $c(0) = 0$, and minimize $c(t_f)$ as a terminal cost. Check
   that the answer does not change.

## Further reading

- H. J. Sussmann and J. C. Willems, "300 years of optimal control: from the
  brachistochrone to the maximum principle", *IEEE Control Systems Magazine*
  17(3), 1997. The history of this lecture and of the whole course, told by
  two of its practitioners.
- H. H. Goldstine, *A History of the Calculus of Variations from the 17th
  through the 19th Century* (Springer, 1980), chapter 1.
- A. E. Bryson and Y.-C. Ho, *Applied Optimal Control* (1969), chapter 2.